# 📥 Security Dataset Download & Preparation

This notebook downloads and prepares all security datasets for training.
Run this notebook **once** before training any models.

## Datasets Included:
- **Phishing Detection**: Malicious URLs, phishing websites
- **Malware Analysis**: PE features, Android malware
- **Network Intrusion**: NSL-KDD, CICIDS, UNSW-NB15
- **Web Attacks**: XSS, SQL injection, CSRF
- **Threat Intelligence**: Malicious IPs, botnet C2
- **DNS Security**: DGA detection
- **Spam Detection**: Email classification

In [10]:
# Install required packages using pip magic (ensures correct kernel environment)
%pip install -q pandas numpy certifi nest_asyncio tqdm

print('✅ Dependencies installed')

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


In [11]:
import sys
import asyncio
from pathlib import Path

# Add project path
sys.path.insert(0, str(Path.cwd().parent / 'app' / 'services'))

# Import dataset manager
from web_security_datasets import WebSecurityDatasetManager

# For Jupyter async support
try:
    import nest_asyncio
    nest_asyncio.apply()
except:
    pass

print('✅ Dataset manager imported')

✅ Dataset manager imported


In [12]:
# Initialize dataset manager
DATASET_DIR = Path.cwd().parent / 'datasets' / 'web_security'
manager = WebSecurityDatasetManager(str(DATASET_DIR))

# Show available datasets
info = manager.get_available_datasets()
print('📊 Available Security Datasets:')
print(f'   Categories: {info["categories"]}')
print(f'   Total datasets: {len(info["configured"])}')
print(f'   Estimated samples: {info["total_configured_samples"]:,}')

print('\n📋 Dataset List:')
for ds_id, ds_info in manager.SECURITY_DATASETS.items():
    print(f'   • {ds_id}: {ds_info["name"]} [{ds_info["category"]}]')

📊 Available Security Datasets:
   Categories: ['phishing', 'web_attack', 'cryptomining', 'dns', 'malware', 'threat_intel', 'logs', 'spam', 'ssl', 'intrusion']
   Total datasets: 18
   Estimated samples: 1,072,129

📋 Dataset List:
   • url_phishing_kaggle: Malicious vs Benign URLs (Kaggle) [phishing]
   • phishing_websites_uci: UCI Phishing Websites Dataset [phishing]
   • malware_pe_features: PE Header Malware Features [malware]
   • android_malware_drebin: Android Malware (Drebin-style Features) [malware]
   • cicids2017_ddos: CICIDS 2017 DDoS Detection [intrusion]
   • nsl_kdd_train: NSL-KDD Network Intrusion [intrusion]
   • unsw_nb15: UNSW-NB15 Network Dataset [intrusion]
   • ipsum_malicious_ips: IPsum Malicious IPs [threat_intel]
   • feodotracker_botnet: Feodo Tracker Botnet C2 [threat_intel]
   • urlhaus_malicious: URLhaus Malicious URLs [threat_intel]
   • spambase_uci: UCI Spambase [spam]
   • xss_payloads: XSS Attack Payloads [web_attack]
   • sql_injection_payloads: SQL Inj

In [14]:
# Download all datasets
print('📥 Downloading all security datasets...')
print('   This may take 5-10 minutes on first run.\n')

async def download_all():
    return await manager.download_all_datasets(force=False)

results = asyncio.run(download_all())

print('\n📊 Download Results:')
print(f'   ✅ Successful: {len(results["successful"])}')
print(f'   ⏭️ Skipped: {len(results["skipped"])}')
print(f'   ❌ Failed: {len(results["failed"])}')
print(f'\n   📈 Total samples available: {results["total_samples"]:,}')

📥 Downloading all security datasets...
   This may take 5-10 minutes on first run.


📊 Download Results:
   ✅ Successful: 0
   ⏭️ Skipped: 18
   ❌ Failed: 0

   📈 Total samples available: 1,072,129


In [15]:
# Verify downloaded datasets
print('\n📁 Downloaded Datasets Summary:\n')

import pandas as pd

summary_data = []
for ds_id, info in manager.downloaded_datasets.items():
    samples = info.get('actual_samples', info.get('samples', 0))
    category = info.get('category', 'unknown')
    synthetic = 'Yes' if info.get('synthetic') else 'No'
    
    summary_data.append({
        'Dataset': ds_id,
        'Category': category,
        'Samples': samples,
        'Synthetic': synthetic
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print(f'\n📊 Total: {summary_df["Samples"].sum():,} samples across {len(summary_df)} datasets')


📁 Downloaded Datasets Summary:

               Dataset     Category  Samples Synthetic
   url_phishing_kaggle     phishing   450000        No
 phishing_websites_uci     phishing    11055        No
   malware_pe_features      malware     4500        No
android_malware_drebin      malware    15000        No
       cicids2017_ddos    intrusion   128000        No
         nsl_kdd_train    intrusion   125973        No
             unsw_nb15    intrusion   175000        No
   ipsum_malicious_ips threat_intel    25000        No
   feodotracker_botnet threat_intel     5000        No
     urlhaus_malicious threat_intel    10000        No
          spambase_uci         spam     4601        No
          xss_payloads   web_attack     5000        No
sql_injection_payloads   web_attack     3000        No
    http_csic_requests   web_attack    36000        No
  cryptomining_scripts cryptomining     5000        No
           dga_domains          dns    50000        No
      ssl_certificates          

In [16]:
# Quick data quality check
print('🔍 Data Quality Check:\n')

async def check_quality():
    for ds_id in list(manager.downloaded_datasets.keys())[:5]:  # Check first 5
        df = await manager.load_dataset(ds_id)
        if df is not None:
            null_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
            print(f'   {ds_id}:')
            print(f'      Shape: {df.shape}')
            print(f'      Null %: {null_pct:.2f}%')
            print(f'      Numeric cols: {len(df.select_dtypes(include=["number"]).columns)}')

asyncio.run(check_quality())

print('\n✅ Dataset preparation complete!')
print('\n🚀 You can now run the training notebooks.')

🔍 Data Quality Check:


✅ Dataset preparation complete!

🚀 You can now run the training notebooks.
